<a href="https://colab.research.google.com/github/MElsdk-lab/Biochar_forest_estimation/blob/main/Main_Notebook_Biochar_Forest_Estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================
# MAIN NOTEBOOK — Biochar Forest Estimation Pipeline
# University of Pittsburgh | Biochar Feedstock Methodology
# ============================================================

In [48]:
# ── CELL 1: Install Libraries ─────────────────────────────────────────────────
!pip install -q earthengine-api geemap

In [67]:
# ── CELL 2: Mount Google Drive + define paths ─────────────
from google.colab import drive
drive.mount('/content/drive')

# Drive paths — for GEE raw exports
DRIVE_FOLDER = '/content/drive/MyDrive/BiocharProject/'
GEE_FOLDER = '/content/drive/MyDrive/BiocharProject/GEE_exports/'
FAO_FOLDER   = '/content/drive/MyDrive/BiocharProject/FAO_data/'


# Repo paths — for final cleaned results
REPO_FOLDER  = '/content/Biochar_forest_estimation/'
DATA_FOLDER  = REPO_FOLDER + 'data/'

print('✅ Google Drive mounted')
print('✅ Paths defined')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted
✅ Paths defined


In [50]:
# ── CELL 3: Authenticate GEE ──────────────────────────────────────────────────
import ee
import os
os.environ['GOOGLE_MAPS_API_KEY'] = ''

ee.Authenticate()   # ← uncomment ONLY if token expired
ee.Initialize(project='spry-blade-487218-n0')

print('✅ GEE connected')

✅ GEE connected


In [53]:
# ── CELL 4: Clone GitHub Repo ─────────────────────────────
%cd /content/
!rm -rf /content/Biochar_forest_estimation

import getpass, subprocess
PAT = getpass.getpass('Enter your GitHub PAT: ')

result = subprocess.run(
    f'git clone https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git',
    shell=True, capture_output=True, text=True
)

if result.returncode == 0:
    %cd /content/Biochar_forest_estimation/
    print('✅ Repo cloned')
else:
    print('❌ Clone failed')
    print(result.stderr)
print(os.getcwd())
print(os.listdir('.'))

/content
Enter your GitHub PAT: ··········
/content/Biochar_forest_estimation
✅ Repo cloned


In [55]:
 #── CELL 5: Run Notebook 1 — GEE Computation ─────────────────────────────────
%run Notebook_1_GEE_Forest_Area_Computation.ipynb

print('✅ Notebook 1 complete — tasks submitted to GEE')

/usr/local/lib/python3.12/dist-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


✅ Libraries imported
✅ Data config loaded
✅ Hansen GFC 2024 loaded
✅ GLC FCS30D annual loaded
✅ GLC FCS30D five-year loaded
✅ treecover2000 masked
✅ GLC FCS30D 2000 loaded
✅ GLC forest mask created (classes 51-92)
✅ 10 forest classes defined
✅ prepare_forest_collection() defined
✅ export_forest_area() defined
✅ prepare_states_forest_collection() defined
✅ export_states_forest_area() defined
✅ 50 US states defined
── Submitting country tasks ──
✅ Task submitted for threshold 10%
✅ Task submitted for threshold 20%
✅ Task submitted for threshold 30%
✅ Task submitted for threshold 40%
✅ Task submitted for threshold 50%

── Submitting state tasks ──
✅ Task submitted for threshold 10%
✅ Task submitted for threshold 20%
✅ Task submitted for threshold 30%
✅ Task submitted for threshold 40%
✅ Task submitted for threshold 50%
── Country tasks ──
  forest_area_10: READY
  forest_area_20: READY
  forest_area_30: READY
  forest_area_40: READY
  forest_area_50: READY

── State tasks ──
  states_fore

In [59]:
# ── CELL 6: Monitor GEE Tasks ─────────────────────────────────────────────────
# ⚠️ Run this cell manually — repeat until all tasks show COMPLETED
import time

for i in range(30):                          # check up to 30 times
    print(f'\n── Check {i+1} ──')
    all_done = True

    print('Country tasks:')
    for task in country_tasks:
        status = task.status()
        print(f"  {status['description']}: {status['state']}")
        if status['state'] not in ['COMPLETED', 'FAILED']:
            all_done = False

    print('State tasks:')
    for task in state_tasks:
        status = task.status()
        print(f"  {status['description']}: {status['state']}")
        if status['state'] not in ['COMPLETED', 'FAILED']:
            all_done = False

    if all_done:
        print('\n✅ All tasks completed!')
        break

    time.sleep(60)


── Check 1 ──
Country tasks:
  forest_area_10: COMPLETED
  forest_area_20: COMPLETED
  forest_area_30: COMPLETED
  forest_area_40: COMPLETED
  forest_area_50: COMPLETED
State tasks:
  states_forest_area_10: COMPLETED
  states_forest_area_20: COMPLETED
  states_forest_area_30: COMPLETED
  states_forest_area_40: COMPLETED
  states_forest_area_50: COMPLETED

✅ All tasks completed!


attention to the name of the notebook 2
it should be fixed

In [77]:
# ── CELL 7: export GEE exports created from Notebook 1 and stored on Google drive to github────────────────────────────────────────
import shutil
print(os.listdir(GEE_FOLDER))

# copy raw GEE exports from Drive to repo data folder
files = [
    'forest_area_10.csv',    'forest_area_20.csv', 'forest_area_30.csv','forest_area_40.csv',  'forest_area_50.csv',
    'states_forest_area_10.csv', 'states_forest_area_20.csv','states_forest_area_30.csv', 'states_forest_area_40.csv',  'states_forest_area_50.csv'
        ]

for f in files:
    shutil.copy(GEE_FOLDER + f, DATA_FOLDER + f)
    print(f'✅ Copied {f} to repo')

✅ Copied forest_area_10.csv to repo
✅ Copied forest_area_20.csv to repo
✅ Copied forest_area_30.csv to repo
✅ Copied forest_area_40.csv to repo
✅ Copied forest_area_50.csv to repo
✅ Copied states_forest_area_10.csv to repo
✅ Copied states_forest_area_20.csv to repo
✅ Copied states_forest_area_30.csv to repo
✅ Copied states_forest_area_40.csv to repo
✅ Copied states_forest_area_50.csv to repo


In [79]:
# ── CELL 8: Run Notebook 2 — Analysis ────────────────────────────────────────
%run Notebook_2_Forest_Area_Analysis.ipynb
#test if push and commit work

print('✅ Notebook 2 complete — results ready')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Libraries imported
✅ Data config loaded
       country  threshold  total_forest_area     region  subregion
0  Afghanistan         10           0.427305  Near East  West Asia
1  Afghanistan         20           0.287657  Near East  West Asia
2  Afghanistan         30           0.239637  Near East  West Asia
       country     region                 subregion   2000
0  Afghanistan  Near East                 West Asia   1209
1      Albania     Europe            Eastern Europe    769
2      Algeria  Near East  South/East Mediterranean   1708
3      Andorra     Europe            Western Europe     17
4       Angola     Africa           Southern Africa  76013
       country    1990    2000    2010    2015    2020    2025     region  \
0  Afghanistan   1.209   1.209   1.209   1.209   1.209   1.209  Near East   
1      Albania   0.789   0.769   0.782   0.797   0.94

In [81]:
# ── CELL 9: Push Results to GitHub ───────────────────────────────────────────
import subprocess

# pull first to get latest changes
!git pull https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git main

# then push
!git add .
!git commit -m "update forest area results"

result = subprocess.run(
    f'git push https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git main',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print('✅ Results pushed to GitHub')
else:
    print('❌ Push failed')
    print(result.stderr)

From https://github.com/MElsdk-lab/Biochar_forest_estimation
 * branch            main       -> FETCH_HEAD
Already up to date.
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
✅ Results pushed to GitHub
